# 03 · Gold layer — star schema for the four questions

![Star schema](docs/star_schema.png)

### Fact table and grain

**`gold.fact_rating`** — one row per **user × movie**: a user's current star rating of a movie, on the
UTC date it was given. Composite natural key `(user_key, movie_key)`.

| Column | Role |
|---|---|
| `user_key` | FK → `dim_user` |
| `movie_key` | FK → `dim_movie` |
| `date_key` | FK → `dim_date` (`YYYYMMDD`) |
| `rating` | measure: stars, 0.5–5.0 (averaged in every question) |
| `is_positive` | measure: 1 if rating ≥ 4.0 else 0 — its average is the share of positive ratings |

Counts of ratings are `COUNT(*)` over the fact; no separate count column is stored.

### Dimensions — each exists because a question needs it

| Dimension | Key | Used by | Main attributes |
|---|---|---|---|
| `dim_date` | `date_key` | Q1 (trend by year) | full_date, year, quarter, month, month_name, year_month, day_name, is_weekend |
| `dim_movie` | `movie_key` | Q3 (release decades) | movie_id, title, release_year, release_decade, genre_count |
| `dim_genre` | `genre_key` | Q2 (genre ranking) | genre_name, is_content_genre |
| `dim_user` | `user_key` | Q4 (user segments) | user_id, lifetime_ratings, activity_segment, segment_order |

**Why a bridge table for genres.** A movie has 0–many genres, so genre cannot be a single foreign key on the
fact. `bridge_movie_genre (movie_key, genre_key, allocation_weight)` resolves the many-to-many relationship in
the standard Kimball way without copying genre text into the fact or exploding the fact table. In Q2 each
rating counts once **in every genre of its movie** (the natural reading of "average rating of Drama
movies"); `allocation_weight = 1 / genre_count` is stored so weighted totals can be produced when
genre shares must add up to 100%.

Keys: every dimension gets an integer surrogate key generated in a stable order (by the natural id).
DuckLake does not enforce PRIMARY/FOREIGN KEY constraints, so key uniqueness and referential integrity are
**checked explicitly** at the end of this notebook and the run fails if any check fails.

Gold is rebuilt from Silver on every run (drop-and-recreate inside a transaction).

In [ ]:
import pandas as pd
from IPython.display import display

from lakehouse import connect, rebuild_table

con = connect()
counts = {}

## 1. `dim_date`

A continuous calendar spanning the first to the last rating date, so days with no ratings still exist (a
gap is then visible as a zero, not a missing row).

In [ ]:
first_day, last_day = con.sql("SELECT min(rating_date), max(rating_date) FROM silver.ratings").fetchone()
print("Rating dates span", first_day, "→", last_day)

counts["gold.dim_date"] = rebuild_table(con, "gold.dim_date", f'''
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_key,
       CAST(d AS DATE)                         AS full_date,
       year(d)                                 AS year,
       quarter(d)                              AS quarter,
       month(d)                                AS month,
       strftime(d, '%B')                       AS month_name,
       strftime(d, '%Y-%m')                    AS year_month,
       dayname(d)                              AS day_name,
       isodow(d) IN (6, 7)                     AS is_weekend
FROM generate_series(TIMESTAMP '{first_day}', TIMESTAMP '{last_day}', INTERVAL 1 DAY) AS t(d)
''')
display(con.sql("SELECT * FROM gold.dim_date ORDER BY date_key LIMIT 3").df())

## 2. `dim_movie`, `dim_genre` and `bridge_movie_genre`

`release_decade` is a label (`'1990s'`) for display; `release_decade_start` (`1990`) sorts it.
Movies whose title carries no year get `'Unknown'`.
`is_content_genre` is false for `IMAX`, which MovieLens lists among genres but is a projection format, not a
kind of story — Q2 excludes it.

In [ ]:
counts["gold.dim_movie"] = rebuild_table(con, "gold.dim_movie", '''
SELECT CAST(row_number() OVER (ORDER BY movie_id) AS INTEGER) AS movie_key,
       movie_id,
       title_clean AS title,
       release_year,
       CASE WHEN release_year IS NULL THEN 'Unknown'
            ELSE CAST(release_year // 10 * 10 AS VARCHAR) || 's' END AS release_decade,
       release_year // 10 * 10 AS release_decade_start,
       genre_count
FROM silver.movies
''')

counts["gold.dim_genre"] = rebuild_table(con, "gold.dim_genre", '''
SELECT CAST(row_number() OVER (ORDER BY genre) AS INTEGER) AS genre_key,
       genre AS genre_name,
       genre <> 'IMAX' AS is_content_genre
FROM (SELECT DISTINCT genre FROM silver.movie_genres)
''')

counts["gold.bridge_movie_genre"] = rebuild_table(con, "gold.bridge_movie_genre", '''
SELECT m.movie_key,
       g.genre_key,
       1.0 / m.genre_count AS allocation_weight
FROM silver.movie_genres mg
JOIN gold.dim_movie m ON m.movie_id = mg.movie_id
JOIN gold.dim_genre g ON g.genre_name = mg.genre
''')
display(con.sql("SELECT * FROM gold.dim_movie ORDER BY movie_key LIMIT 5").df())
display(con.sql("SELECT * FROM gold.dim_genre ORDER BY genre_key").df())

## 3. `dim_user`

MovieLens users are anonymous, so the only meaningful user attribute is behavioural: how many movies they
have rated. Users are segmented by lifetime rating count (Q4):

| Segment | Lifetime ratings |
|---|---|
| Casual | fewer than 50 |
| Regular | 50–199 |
| Heavy | 200–999 |
| Power | 1,000 or more |

The segment is recalculated on every rebuild (type-1 attribute), so it always reflects the user's full history.

In [ ]:
counts["gold.dim_user"] = rebuild_table(con, "gold.dim_user", '''
WITH u AS (
    SELECT user_id, count(*) AS n, min(rating_date) AS first_rating_date
    FROM silver.ratings GROUP BY user_id
)
SELECT CAST(row_number() OVER (ORDER BY user_id) AS INTEGER) AS user_key,
       user_id,
       first_rating_date,
       n AS lifetime_ratings,
       CASE WHEN n < 50   THEN 'Casual (<50)'
            WHEN n < 200  THEN 'Regular (50–199)'
            WHEN n < 1000 THEN 'Heavy (200–999)'
            ELSE 'Power (1000+)' END AS activity_segment,
       CASE WHEN n < 50 THEN 1 WHEN n < 200 THEN 2 WHEN n < 1000 THEN 3 ELSE 4 END AS segment_order
FROM u
''')
display(con.sql('''
    SELECT activity_segment, count(*) AS users FROM gold.dim_user
    GROUP BY activity_segment, segment_order ORDER BY segment_order
''').df())

## 4. `fact_rating`

Built from Silver by swapping natural ids for surrogate keys. Only the keys and the two measures are kept —
titles, genres and user attributes live in the dimensions, not in the fact.

In [ ]:
counts["gold.fact_rating"] = rebuild_table(con, "gold.fact_rating", '''
SELECT u.user_key,
       m.movie_key,
       CAST(strftime(r.rating_date, '%Y%m%d') AS INTEGER) AS date_key,
       r.rating,
       CASE WHEN r.rating >= 4.0 THEN 1 ELSE 0 END AS is_positive
FROM silver.ratings r
JOIN gold.dim_user  u ON u.user_id  = r.user_id
JOIN gold.dim_movie m ON m.movie_id = r.movie_id
''')
display(con.sql("SELECT * FROM gold.fact_rating LIMIT 5").df())
display(pd.DataFrame({"table": counts.keys(), "rows": counts.values()}))

## 5. Key and referential-integrity checks

Every check must return 0 problems (or `true`). Inner joins in the fact build could silently drop rows, so the
fact is also reconciled against `silver.ratings`.

In [ ]:
checks = con.sql('''
SELECT * FROM (VALUES
 ('PK unique: dim_date.date_key',
     (SELECT count(*) - count(DISTINCT date_key) FROM gold.dim_date)),
 ('PK unique: dim_movie.movie_key',
     (SELECT count(*) - count(DISTINCT movie_key) FROM gold.dim_movie)),
 ('PK unique: dim_genre.genre_key',
     (SELECT count(*) - count(DISTINCT genre_key) FROM gold.dim_genre)),
 ('PK unique: dim_user.user_key',
     (SELECT count(*) - count(DISTINCT user_key) FROM gold.dim_user)),
 ('Grain unique: fact (user_key, movie_key)',
     (SELECT count(*) FROM (SELECT user_key, movie_key FROM gold.fact_rating
                            GROUP BY ALL HAVING count(*) > 1))),
 ('Orphan FK: fact.user_key → dim_user',
     (SELECT count(*) FROM gold.fact_rating f ANTI JOIN gold.dim_user d USING (user_key))),
 ('Orphan FK: fact.movie_key → dim_movie',
     (SELECT count(*) FROM gold.fact_rating f ANTI JOIN gold.dim_movie d USING (movie_key))),
 ('Orphan FK: fact.date_key → dim_date',
     (SELECT count(*) FROM gold.fact_rating f ANTI JOIN gold.dim_date d USING (date_key))),
 ('Orphan FK: bridge.movie_key → dim_movie',
     (SELECT count(*) FROM gold.bridge_movie_genre b ANTI JOIN gold.dim_movie d USING (movie_key))),
 ('Orphan FK: bridge.genre_key → dim_genre',
     (SELECT count(*) FROM gold.bridge_movie_genre b ANTI JOIN gold.dim_genre d USING (genre_key))),
 ('Fact rows missing vs silver.ratings',
     (SELECT (SELECT count(*) FROM silver.ratings) - (SELECT count(*) FROM gold.fact_rating))),
 ('dim_user lifetime_ratings ≠ fact rows per user',
     (SELECT count(*) FROM gold.dim_user u
      LEFT JOIN (SELECT user_key, count(*) AS n FROM gold.fact_rating GROUP BY 1) f USING (user_key)
      WHERE coalesce(f.n, 0) <> u.lifetime_ratings))
) AS t(check_name, problems)
''').df()
checks["passed"] = checks["problems"] == 0
display(checks)
assert checks["passed"].all(), "Gold integrity checks failed"
print("All Gold key and referential-integrity checks passed ✔")

In [ ]:
con.close()
print("Gold layer complete.")